<a href="https://colab.research.google.com/github/fadiaputriananda/seismology-portfolio/blob/main/seismic_monitoring_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install Libary**

In [4]:
!pip install pandas
!pip install plotly
!pip install gspread
!pip install gspread_dataframe

**Take the USGS Data**

In [63]:
import pandas as pd
from datetime import datetime, timedelta

start_date_str = '2026-01-01'
today_str = datetime.today().strftime('%Y-%m-%d')

start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
end_date = datetime.strptime(today_str, '%Y-%m-%d')

url = f"https://earthquake.usgs.gov/fdsnws/event/1/query.csv?starttime={start_date_str}&endtime={today_str}&minmagnitude=3"
df = pd.read_csv(url)

df.head()

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2026-11-26T23:52:59.392Z,54.0870,-169.4030,13.400,3.80,ml,34.0,105.0,0.9000,0.80,...,2026-11-27T01:58:14.068Z,"132 km NNW of Nikolski, Alaska",earthquake,3.70,3.6473,0.300000,25.0,reviewed,ak,ak
1,2026-11-26T23:31:07.918Z,35.6479,29.8422,31.156,4.10,mb,27.0,135.0,1.0990,0.57,...,2026-11-27T00:01:55.040Z,"64 km SSE of Kaş, Turkey",earthquake,2.77,7.2640,0.213000,6.0,reviewed,us,us
2,2026-11-26T22:21:44.644Z,63.1000,-148.4690,73.300,3.10,ml,98.0,17.0,0.4000,0.80,...,2026-11-26T22:51:01.040Z,"40 km SE of Cantwell, Alaska",earthquake,1.90,3.3218,0.200000,73.0,automatic,ak,ak
3,2026-11-26T21:43:20.452Z,69.5970,-144.0320,15.100,3.30,ml,56.0,118.0,0.1000,1.30,...,2026-11-26T22:22:14.040Z,"61 km SSW of Kaktovik, Alaska",earthquake,6.20,3.4315,0.200000,30.0,automatic,ak,ak
4,2026-11-26T21:33:42.450Z,18.8610,-67.8740,21.120,3.21,md,10.0,207.0,0.5914,0.24,...,2026-11-26T21:58:14.350Z,"63 km ENE of Punta Cana, Dominican Republic",earthquake,1.14,19.3700,0.060322,10.0,reviewed,pr,pr


In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19347 entries, 0 to 19346
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   time             19347 non-null  object 
 1   latitude         19347 non-null  float64
 2   longitude        19347 non-null  float64
 3   depth            19347 non-null  float64
 4   mag              19347 non-null  float64
 5   magType          19347 non-null  object 
 6   nst              19346 non-null  float64
 7   gap              19343 non-null  float64
 8   dmin             19337 non-null  float64
 9   rms              19346 non-null  float64
 10  net              19347 non-null  object 
 11  id               19347 non-null  object 
 12  updated          19347 non-null  object 
 13  place            19346 non-null  object 
 14  type             19347 non-null  object 
 15  horizontalError  19216 non-null  float64
 16  depthError       19345 non-null  float64
 17  magError    

**Data Cleaning**

In [64]:
## Pick Earthquake Type Only and apply to df
df = df[df['type'] == 'earthquake'].copy()
df.head()

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2026-11-26T23:52:59.392Z,54.0870,-169.4030,13.400,3.80,ml,34.0,105.0,0.9000,0.80,...,2026-11-27T01:58:14.068Z,"132 km NNW of Nikolski, Alaska",earthquake,3.70,3.6473,0.300000,25.0,reviewed,ak,ak
1,2026-11-26T23:31:07.918Z,35.6479,29.8422,31.156,4.10,mb,27.0,135.0,1.0990,0.57,...,2026-11-27T00:01:55.040Z,"64 km SSE of Kaş, Turkey",earthquake,2.77,7.2640,0.213000,6.0,reviewed,us,us
2,2026-11-26T22:21:44.644Z,63.1000,-148.4690,73.300,3.10,ml,98.0,17.0,0.4000,0.80,...,2026-11-26T22:51:01.040Z,"40 km SE of Cantwell, Alaska",earthquake,1.90,3.3218,0.200000,73.0,automatic,ak,ak
3,2026-11-26T21:43:20.452Z,69.5970,-144.0320,15.100,3.30,ml,56.0,118.0,0.1000,1.30,...,2026-11-26T22:22:14.040Z,"61 km SSW of Kaktovik, Alaska",earthquake,6.20,3.4315,0.200000,30.0,automatic,ak,ak
4,2026-11-26T21:33:42.450Z,18.8610,-67.8740,21.120,3.21,md,10.0,207.0,0.5914,0.24,...,2026-11-26T21:58:14.350Z,"63 km ENE of Punta Cana, Dominican Republic",earthquake,1.14,19.3700,0.060322,10.0,reviewed,pr,pr


In [65]:
import pytz

## Convert 'time' column to datetime, convert to WIB, strip timezone, and save it directly back to df
df['time'] = pd.to_datetime(df['time'])
wib = pytz.timezone('Asia/Jakarta')
df['time'] = df['time'].dt.tz_convert(wib).dt.tz_localize(None)

# Sort the DataFrame chronologically by time
df = df.sort_values(by='time', ascending=True)
df['time'].head()

,time
19346,2026-01-01 07:03:02.325
19345,2026-01-01 07:46:26.988
19344,2026-01-01 08:25:20.926
19343,2026-01-01 08:53:01.809
19342,2026-01-01 09:12:59.818


In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19234 entries, 19346 to 0
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   time             19234 non-null  datetime64[ns]
 1   latitude         19234 non-null  float64       
 2   longitude        19234 non-null  float64       
 3   depth            19234 non-null  float64       
 4   mag              19234 non-null  float64       
 5   magType          19234 non-null  object        
 6   nst              19233 non-null  float64       
 7   gap              19233 non-null  float64       
 8   dmin             19227 non-null  float64       
 9   rms              19233 non-null  float64       
 10  net              19234 non-null  object        
 11  id               19234 non-null  object        
 12  updated          19234 non-null  object        
 13  place            19233 non-null  object        
 14  type             19234 non-null  object    

In [67]:
#Rename and Choose some columns
df = df.rename(columns={'time': 'Time',
                       'latitude': 'Latitude',
                       'longitude': 'Longitude',
                       'mag': 'Magnitude',
                       'place': 'Place',
                       'depth': 'Depth'})
df = df[['Time','Place','Latitude','Longitude','Magnitude','Depth']]
df.head()

,Time,Place,Latitude,Longitude,Magnitude,Depth
19346,2026-01-01 07:03:02.325,"100 km N of Yakutat, Alaska",60.4463,-139.6346,3.2,5.000
19345,2026-01-01 07:46:26.988,"262 km SE of Chiniak, Alaska",56.3007,-148.7003,3.9,10.000
19344,2026-01-01 08:25:20.926,South Sandwich Islands region,-56.1770,-28.1811,4.5,118.207
19343,2026-01-01 08:53:01.809,southeast Indian Ridge,-45.6117,96.3592,6.0,10.000
19342,2026-01-01 09:12:59.818,"168 km SSE of Vilyuchinsk, Russia",51.5185,159.3017,4.6,46.115


In [71]:
df.Time.sort_values(ascending=True)

df.Time.tail()

,Time
4,2026-11-27 04:33:42.450
3,2026-11-27 04:43:20.452
2,2026-11-27 05:21:44.644
1,2026-11-27 06:31:07.918
0,2026-11-27 06:52:59.392


In [73]:
#Make Country Columns from dataframe
# Using .str.split() handles any missing/NaN values safely
df['Country'] = df['Place'].str.split(',').str[-1]
df['Country'] = df['Country'].str.strip()
df.head()

,Time,Place,Latitude,Longitude,Magnitude,Depth,Country
19346,2026-01-01 07:03:02.325,"100 km N of Yakutat, Alaska",60.4463,-139.6346,3.2,5.000,Alaska
19345,2026-01-01 07:46:26.988,"262 km SE of Chiniak, Alaska",56.3007,-148.7003,3.9,10.000,Alaska
19344,2026-01-01 08:25:20.926,South Sandwich Islands region,-56.1770,-28.1811,4.5,118.207,South Sandwich Islands region
19343,2026-01-01 08:53:01.809,southeast Indian Ridge,-45.6117,96.3592,6.0,10.000,southeast Indian Ridge
19342,2026-01-01 09:12:59.818,"168 km SSE of Vilyuchinsk, Russia",51.5185,159.3017,4.6,46.115,Russia


In [74]:
df.describe()

,Time,Latitude,Longitude,Magnitude,Depth
count,19234,19234.000000,19234.000000,19234.000000,19234.000000
mean,2026-06-14 00:07:57.321965056,13.912976,17.925263,4.247580,67.463838
min,2026-01-01 07:03:02.325000,-84.493200,-179.999300,3.000000,-3.430000
25%,2026-03-28 16:14:48.790249984,-8.379375,-98.695575,4.000000,10.000000
50%,2026-06-15 21:37:45.000499968,14.122450,68.789650,4.300000,18.630000
75%,2026-08-29 02:53:01.699749888,40.028400,138.969725,4.600000,67.421250
max,2026-11-27 06:52:59.392000,86.723600,179.999100,7.800000,683.578000
std,NaN,30.279649,129.971213,0.625594,118.222668


**Data Analysis and Data Visualization**

In [75]:
#Histogram Magnitude

import plotly.express as px
fig = px.histogram(
    df,
    x = "Magnitude",
    nbins = 18,
    title = "Distribution of Earthquake Magnitude"
)
fig.show()


Based on the distribution of earthquake magnitude, the histogram showcases a right-skewed distribution, which is highly typical for earthquake frequency data. The vast majority of the recorded seismic events are concentrated in the lower magnitude spectrum, specifically peaking between 4.2 and 4.6 magnitudes (classified as Light Earthquakes). As the magnitude increases toward moderate and strong levels, the frequency drops exponentially, leaving only a few rare, high-magnitude events above 7.0 at the far right of the scale.

In [76]:
#Scatter Plot Depth vs Magnitude

import plotly.express as px
fig = px.scatter(
    df,
    x = "Depth",
    y = "Magnitude",
    hover_name = "Place",
    title = "Depth vs Magnitude",
)
fig.show()

Based on the scatter plot of the depth value vs the magnitude value, we can simply say that there is no strong linear correlation between earthquake depth and its magnitude. However, the data shows a high concentration of seismic events at shallow depths (0-150 km) with magnitudes between 4.0 and 5.5. Additionally, a distinct cluster of deep-focus earthquakes is visible around the 500-650 km depth mark.

**Interactive Earthquake Map**

In [77]:
import plotly.express as px

df['Depth'] = df['Depth'].abs() # Convert negative depth values to positive

fig = px.scatter_map(
    df,
    lat = 'Latitude',
    lon = 'Longitude',
    size = 'Depth',
    color = 'Magnitude',
    hover_name = 'Place',
    title = "Global Earthquake Monitoring Map",
    zoom = 1
)
fig.show()

Based on the interactive global earthquake map, it can be concluded that most earthquakes occur in the Pacific Ring of Fire, a zone with a high risk of earthquakes because it is where the world’s major tectonic plates converge, particularly in Indonesia, the Philippines, Japan, Tonga, & Vanuatu.

**Save Dataset**

In [78]:
#Save dataset on Google Spread Sheets in real-time

#Connect Google Colab to Google Spreadsheets
from google.colab import auth
auth.authenticate_user()

In [79]:
#Import Library
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe

In [80]:
#Authorized Google Sheets
creds, _ = default()
gc = gspread.authorize(creds)

In [81]:
#Open the Google Sheets (Worksheet File)
spreadsheet = gc.open("Seismic Dashboard Data")
worksheet = spreadsheet.sheet1

In [82]:
#Push the Data Frame to the Google Sheets
worksheet.clear()
set_with_dataframe(worksheet, df)

In [84]:
#save dataset in local device
df.to_csv("earthquake_data.csv", index = False)